# Step-by-Step Neural Network Training (Matt Mazur Example)

This notebook reproduces the exact neural network configuration described in the article ["A Step by Step Backpropagation Example"](https://mattmazur.com/2015/03/17/a-step-by-step-backpropagation-example/).

It uses **PyTorch** to define and train the network, capturing all relevant values like weights, biases, activations, and outputs in a structured log that can be exported and reused.

## 1. Import libraries
We begin by importing the required libraries for neural network construction, training, and data logging.

In [27]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd

## 2. Define the neural network
This class defines the architecture of our neural network with manually initialized weights and biases. The structure is:
- 2 input neurons
- 2 hidden neurons with sigmoid activation
- 2 output neurons with sigmoid activation

In [28]:
class FixedInitNN(nn.Module):
    def __init__(self):
        super(FixedInitNN, self).__init__()
        self.input = nn.Linear(2, 2)
        self.output = nn.Linear(2, 2)
        self.sigmoid = nn.Sigmoid()

        with torch.no_grad():
            self.input.weight.copy_(torch.tensor([[0.15, 0.20], [0.25, 0.30]]))
            self.input.bias.copy_(torch.tensor([0.35, 0.35]))
            self.output.weight.copy_(torch.tensor([[0.40, 0.45], [0.50, 0.55]]))
            self.output.bias.copy_(torch.tensor([0.60, 0.60]))

    def forward(self, x):
        self.hidden_input = self.input(x)
        self.hidden_output = self.sigmoid(self.hidden_input)
        self.final_input = self.output(self.hidden_output)
        self.final_output = self.sigmoid(self.final_input)
        return self.final_output

## 3. Define the input and target data
We use the same input and expected output as in the article, based on a single XOR example.

In [29]:
X = torch.tensor([[0.05, 0.10]])
Y = torch.tensor([[0.01, 0.99]])
model = FixedInitNN()
criterion = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=0.5)

## 4. Perform N training step
We do a full forward pass, calculate the loss, backpropagate the gradients, and update the weights. All relevant data is logged in a structured way.

In [30]:
n = 20000
log = []
for epoch in range(n):
    optimizer.zero_grad()
    output = model(X)
    loss = criterion(output, Y)
    loss.backward()
    optimizer.step()

    log.append({
        'epoch': epoch,
        'loss': loss.item(),
        'w_in_0_0': model.input.weight.data[0][0].item(),
        'w_in_0_1': model.input.weight.data[0][1].item(),
        'w_in_1_0': model.input.weight.data[1][0].item(),
        'w_in_1_1': model.input.weight.data[1][1].item(),
        'b_in_0': model.input.bias.data[0].item(),
        'b_in_1': model.input.bias.data[1].item(),
        'h_0': model.hidden_output[0][0].item(),
        'h_1': model.hidden_output[0][1].item(),
        'w_out_0_0': model.output.weight.data[0][0].item(),
        'w_out_0_1': model.output.weight.data[0][1].item(),
        'w_out_1_0': model.output.weight.data[1][0].item(),
        'w_out_1_1': model.output.weight.data[1][1].item(),
        'b_out_0': model.output.bias.data[0].item(),
        'b_out_1': model.output.bias.data[1].item(),
        'y_0': output[0][0].item(),
        'y_1': output[0][1].item()
    })

## 5. Save the log to CSV
We now save the recorded weights, biases, activations, and predictions into a CSV file that can later be used for analysis or visualizations.

In [31]:
df = pd.DataFrame(log)
df.to_csv("training_step_" + str(n) + ".csv", index=False)
print("Log saved to 'training_step_" + str(n) + ".csv'.")

Log saved to 'training_step_20000.csv'.
